# 03 — Extraction test

A thin frontend to `src/extract.py` for **visually inspecting** the structured output of a
single live Gemini call, before wiring up the full `data/in/interviews.json` loop.

> **This notebook always makes a real Gemini API call** when you run the cells below.
> `RUN_LIVE_TESTS` does **not** apply here — that flag only gates the live *pytest* in
> `tests/test_extract.py`, never `extract()`. A `503 … high demand` error just means Gemini was
> temporarily busy; rerun shortly.

Run one transcript through `extract()` and print the resulting `PatientLabels` as indented
JSON, so every field (enums, demographics, `referral_pathway`, …) can be checked by eye.

`extract()` loads `GEMINI_API_KEY` from `.env` itself — no extra setup needed.

In [1]:
import logging
import sys

import litellm

sys.path.append("../src")  # kernel cwd is notebooks/ (per %pwd); src/ is one level up

from extract import LOG_DIR, TESTS_OUT, extract

litellm._turn_on_debug()  # verbose LiteLLM logging for debugging API calls (e.g. 503s)

# litellm logs via the stdlib `logging` module (not loguru); attach a file handler to the
# "LiteLLM" logger so the verbose debug stream persists to logs/litellm_debug.log.
logging.getLogger("LiteLLM").addHandler(logging.FileHandler(LOG_DIR / "litellm_debug.log"))

In [2]:
# Synthetic transcript with unambiguous facts (same as tests/test_extract.py).
transcript = (
    "I'm Alex, 38, male. Diagnosed with Crohn's four years ago. After mesalamine and "
    "prednisone failed, my gastroenterologist started me on Humira, which I've taken ever "
    "since and it keeps me in remission."
)

# out_dir=TESTS_OUT keeps this synthetic prediction in data/out/tests, not production.
labels = extract(transcript, "P000", out_dir=TESTS_OUT)
print(labels.model_dump_json(indent=2))

14:19:10 - LiteLLM:DEBUG: utils.py:485 - 

14:19:10 - LiteLLM:DEBUG: utils.py:485 - Request to litellm:
14:19:10 - LiteLLM:DEBUG: utils.py:485 - litellm.completion(model='gemini/gemini-2.5-flash-lite', messages=[{'role': 'system', 'content': "Extract the structured labels from this Crohn's disease patient interview. Use the schema's enum values exactly, and leave a field null/empty when the transcript doesn't support a confident value. Set patient_id to the value given in the message. For referral_pathway, list the patient's journey as an ordered sequence of the canonical PathwayStep values."}, {'role': 'user', 'content': "patient_id: P000\n\nI'm Alex, 38, male. Diagnosed with Crohn's four years ago. After mesalamine and prednisone failed, my gastroenterologist started me on Humira, which I've taken ever since and it keeps me in remission."}], response_format={'type': 'json_object', 'response_schema': {'$defs': {'ComorbidCondition': {'description': 'Independent coexisting diagnoses onl

{
  "patient_id": "P000",
  "churn": false,
  "incomplete_journey": false,
  "demographics": {
    "gender": "male",
    "age": 38
  },
  "biologic_prescribed": true,
  "biologic_taken": true,
  "biologic_not_mentioned": false,
  "biologic_type": "Humira",
  "reasons_for_biologic_prescribed": "DOCTOR_CHOICE",
  "reasons_for_biologic_not_taken": null,
  "comorbid_conditions": [],
  "treatment_records": [
    {
      "name": "mesalamine",
      "treatment_class": "conventional",
      "outcome": "FAILED",
      "reason_stopped": null
    },
    {
      "name": "prednisone",
      "treatment_class": "conventional",
      "outcome": "FAILED",
      "reason_stopped": null
    },
    {
      "name": "Humira",
      "treatment_class": "biologic",
      "outcome": "SUCCESS",
      "reason_stopped": null
    }
  ],
  "treatment_outcome": "SUCCESS",
  "referral_pathway": [
    "symptom_onset",
    "diagnostic_testing",
    "crohns_diagnosis",
    "conventional_therapy",
    "therapy_failed",
   